# Argument realization in an Ojibwe corpus: frequency counts

In this notebook, we will attempt to parse the frequency counts for argument realization, splitting on 3 cases:
1. Animacy: Is there a difference in the ratio of overt animate vs. inanimate arguments?
2. SAPs: Is there a difference in the ratio of overt SAP vs. non-SAP arguments?
3. Obviation: Is there a difference in the ratio of overt obviative vs. proximate arguments? (only 3-3 VTAs)

In [9]:
import os, sys
from pathlib import Path

REPO_ROOT = Path(os.getcwd()).resolve().parents[1] # ../../
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))
    
corpus_path = REPO_ROOT / "data" / "corpus" / "Living_our_Language" / "LoL_section4.xml"


In [ ]:
import xml.etree.ElementTree as ET

root = tree.getroot()

oj_sents = []
for text_ojb in root.findall(".//text_ojb"):
    for sent in text_ojb.findall("sentence"):
        sent_num = sent.get("num")
        # find sent_text node
        sent_text_node = sent.find("sent_text")
        # get actual sentence text
        sent_text = sent_text_node.text.strip() if sent_text_node is not None else None
        oj_sents.append(sent_text)
        print("Sentence num:", sent_num)
        print("Text:", sent_text)
        print("-" * 40)

In [10]:
from grammar_modules.dependency import DEPENDENCY_PATH, parse_dependencies
from treebank_modules.corpus import cg3_to_conllu_batch
from grammar_modules.disambiguation import DISAMBIGUATION_PATH
from grammar_modules.fst import load_fst_parser

# now we actually build up the .conllu file by first parsing dependencies on each sentence,
# and then appending it to the corpus file

# path to the parsed treebank corpus
CORPUS_PATH = REPO_ROOT / "data" / "treebanks" / "opd_all_treebank.conllu"

# fst + grammar paths
FST = load_fst_parser()
DISAMBIG_CG_PATH = REPO_ROOT / "data" / "rules" / "disambiguation.cg3"
DEPENDENCY_CG_PATH = REPO_ROOT / "data" / "rules" / "dependency.cg3"

FST file is /home/ktu/dev/ling/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin


In [ ]:
num_sents = 0

# for each sentence, parse dependencies and append to conllu corpus
for oj in oj_sents:
    # parse the deps
    dep_cg3 = parse_dependencies(
       sentence=oj,
        dependency_grammar=str(DEPENDENCY_PATH),
        disambiguation_grammar=str(DISAMBIGUATION_PATH),
        fst=FST,
        verbose=False
    )
    # convert and append to corpus (auto sent_id)
    cg3_to_conllu_batch(dep_cg3, corpus_path=str(CORPUS_PATH))
    num_sents += 1

print(f"Added {num_sents} sentences to treebank.")

FST file is /Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/data/fst/ojibwe7.fomabin
✓ appended sentence #1 to LoL_section4.conllu
✓ appended sentence #2 to LoL_section4.conllu
✓ appended sentence #3 to LoL_section4.conllu
✓ appended sentence #4 to LoL_section4.conllu
✓ appended sentence #5 to LoL_section4.conllu
✓ appended sentence #6 to LoL_section4.conllu
✓ appended sentence #7 to LoL_section4.conllu
✓ appended sentence #8 to LoL_section4.conllu
✓ appended sentence #9 to LoL_section4.conllu
✓ appended sentence #10 to LoL_section4.conllu
✓ appended sentence #11 to LoL_section4.conllu
✓ appended sentence #12 to LoL_section4.conllu
✓ appended sentence #13 to LoL_section4.conllu
✓ appended sentence #14 to LoL_section4.conllu
✓ appended sentence #15 to LoL_section4.conllu
✓ appended sentence #16 to LoL_section4.conllu
✓ appended sentence #17 to LoL_section4.conllu
✓ appended sentence #18 to LoL_section4.conllu
✓ appended sentence #19 to LoL_section4.conllu
✓ appended sentence #20

NameError: name 'i' is not defined

### Case 1: Animacy

We will start with animacy. First we will define code that can collect the relevant attributes from a `.conllu` formatted treebank, then we will use this to parse animacy counts.

In [11]:
from conllu import parse_incr

animate_nominal = "NA" 
inanimate_nominal = "NI"

animate_subj_marker = {
    "1SgSubj", "InclSubj", "ExclSubj", "2SgSubj", "2PlSubj",
    "3SgProxSubj", "3PlProxSubj", "3SgObvSubj", "3PlObvSubj",
}
animate_obj_marker = {
    "1SgObj", "InclObj", "ExclObj", "2SgObj", "2PlObj",
    "3SgProxObj", "3PlProxObj", "3SgObvObj", "3PlObvObj",
}
inanimate_subj_marker = {"0SgSubj", "0PlSubj", "0SgObvSubj", "0PlObvSubj"}
inanimate_obj_marker = {"0SgObj", "0PlObj", "0SgObvObj", "0PlObvObj"}

animate_subj_count = animate_obj_count = inanimate_subj_count = inanimate_obj_count = 0
animate_subj_verbs_count = animate_obj_verbs_count = inanimate_subj_verbs_count = inanimate_obj_verbs_count = 0

# for each sentence in treebank, iterate over tokens and collect xpos + deprel 
# (FST tags + dependency relation) and append to counts for each given case
with open(CORPUS_PATH, encoding="utf-8") as f:
    for sent in parse_incr(f):
        for tok in sent:
            xpos = tok.get("xpos")
            if not xpos:
                continue

            # nominal counts
            if animate_nominal in xpos and tok["deprel"] == "nsubj":
                animate_subj_count += 1
            if animate_nominal in xpos and tok["deprel"] == "obj":
                animate_obj_count += 1
            if inanimate_nominal in xpos and tok["deprel"] == "nsubj":
                inanimate_subj_count += 1
            if inanimate_nominal in xpos and tok["deprel"] == "obj":
                inanimate_obj_count += 1

            # verbal person/number markers
            if any(m in xpos for m in animate_subj_marker):
                animate_subj_verbs_count += 1
            if any(m in xpos for m in animate_obj_marker):
                animate_obj_verbs_count += 1
            if any(m in xpos for m in inanimate_subj_marker):
                inanimate_subj_verbs_count += 1
            if any(m in xpos for m in inanimate_obj_marker):
                inanimate_obj_verbs_count += 1

print(
    f"Animate subjects (nominals): {animate_subj_count}",
    f"Animate objects (nominals): {animate_obj_count}",
    f"Inanimate subjects (nominals): {inanimate_subj_count}",
    f"Inanimate objects (nominals): {inanimate_obj_count}",
    sep="\n",
)

print(
    f"Total animate subjects (verbs): {animate_subj_verbs_count}",
    f"Total animate objects (verbs):  {animate_obj_verbs_count}",
    f"Total inanimate subjects (verbs): {inanimate_subj_verbs_count}",
    f"Total inanimate objects (verbs): {inanimate_obj_verbs_count}",
    sep="\n",
)


Animate subjects (nominals): 1105
Animate objects (nominals): 743
Inanimate subjects (nominals): 479
Inanimate objects (nominals): 874
Total animate subjects (verbs): 5704
Total animate objects (verbs):  1555
Total inanimate subjects (verbs): 1246
Total inanimate objects (verbs): 1242


In [12]:
# create stats tables
import pandas as pd

stats = [
    {"argument type": "animate subject", "total": animate_subj_verbs_count, "overt": animate_subj_count,},
    {"argument type": "animate object", "total": animate_obj_verbs_count,  "overt": animate_obj_count,},
    {"argument type": "inanimate subject", "total": inanimate_subj_verbs_count, "overt": inanimate_subj_count,},
    {"argument type": "inanimate object", "total": inanimate_obj_verbs_count,  "overt": inanimate_obj_count,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

    argument type  total  overt  covert  overt_ratio
  animate subject   5704   1105    4599     0.193724
   animate object   1555    743     812     0.477814
inanimate subject   1246    479     767     0.384430
 inanimate object   1242    874     368     0.703704


### Case 2: SAPs

Next, we will parse overt SAPs vs non-SAPs. The prediction is that SAPs have a very low rate of overt arguments compared to non-SAPs.

In [13]:
from conllu import parse_incr

personal_pronoun = "PRONPer"
SAP_argument = {"1Sg", "Incl", "Excl", "2Sg", "2Pl"}

SAP_subj_marker = {
    "1SgSubj", "InclSubj", "ExclSubj", "2SgSubj", "2PlSubj",
}
SAP_obj_marker = {
    "1SgObj", "InclObj", "ExclObj", "2SgObj", "2PlObj",
}
non_SAP_subj_marker = {
    "3SgProxSubj", "3PlProxSubj", "3SgObvSubj", "3PlObvSubj", 
    "0SgSubj", "0PlSubj", "0SgObvSubj", "0PlObvSubj"
    }
non_SAP_obj_marker = {
    "3SgProxObj", "3PlProxObj", "3SgObvObj", "3PlObvObj",
    "0SgObj", "0PlObj", "0SgObvObj", "0PlObvObj",
    }


SAP_subj_count = SAP_obj_count = non_SAP_subj_count = non_SAP_obj_count = 0
SAP_subj_verbs_count = SAP_obj_verbs_count = non_SAP_subj_verbs_count = non_SAP_obj_verbs_count = 0

with open(CORPUS_PATH, encoding="utf-8") as f:
    for sent in parse_incr(f):
        for tok in sent:
            xpos = tok.get("xpos")
            if not xpos:
                continue

            if personal_pronoun in xpos and any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "nsubj":
                SAP_subj_count += 1
            if personal_pronoun in xpos and any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "obj":
                SAP_obj_count += 1
            if not any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "nsubj":
                non_SAP_subj_count += 1
            if not any(sap in xpos for sap in SAP_argument) and tok["deprel"] == "obj":
                non_SAP_obj_count += 1

            if any(m in xpos for m in SAP_subj_marker):
                SAP_subj_verbs_count += 1
            if any(m in xpos for m in SAP_obj_marker):
                SAP_obj_verbs_count += 1
            if any(m in xpos for m in non_SAP_subj_marker):
                non_SAP_subj_verbs_count += 1
            if any(m in xpos for m in non_SAP_obj_marker):
                non_SAP_obj_verbs_count += 1


print(
    "---OVERT SAP/non-SAP arguments---",
    f"SAP subjects: {SAP_subj_count}",
    f"SAP objects: {SAP_obj_count}",
    f"non-SAP subjects: {non_SAP_subj_count}",
    f"non-SAP objects: {non_SAP_obj_count}",
    sep="\n",
)

print(
    "---TOTAL SAP/non-SAP arguments---",
    f"SAP subjects: {SAP_subj_verbs_count}",
    f"SAP objects: {SAP_obj_verbs_count}",
    f"non-SAP subjects: {non_SAP_subj_verbs_count}",
    f"non-SAP objects: {non_SAP_obj_verbs_count}",
    sep="\n",
)


# TODO: I get 76 total arguments here, but 74 in the word_order.ipynb... 
#       need to figure out why the discrepancy. probably the logic doesn't match up 100% somewhere

---OVERT SAP/non-SAP arguments---
SAP subjects: 64
SAP objects: 9
non-SAP subjects: 1428
non-SAP objects: 1362
---TOTAL SAP/non-SAP arguments---
SAP subjects: 2377
SAP objects: 248
non-SAP subjects: 4573
non-SAP objects: 2549


In [15]:
# create stats tables
import pandas as pd

stats = [
    {"argument type": "SAP subjects", "total": SAP_subj_verbs_count, "overt": SAP_subj_count,},
    {"argument type": "SAP objects", "total": SAP_obj_verbs_count,  "overt": SAP_obj_count,},
    {"argument type": "non-SAP subjects", "total": non_SAP_subj_verbs_count,  "overt": non_SAP_subj_count,},
    {"argument type": "non-SAP objects", "total": non_SAP_obj_verbs_count, "overt": non_SAP_obj_count,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

   argument type  total  overt  covert  overt_ratio
    SAP subjects   2377     64    2313     0.026925
     SAP objects    248      9     239     0.036290
non-SAP subjects   4573   1428    3145     0.312268
 non-SAP objects   2549   1362    1187     0.534327


### Case 3: Obviation

We check counts only on 3-3 verbs.


In [ ]:
from conllu import parse_incr

prox_argument = {"ProxSg", "ProxPl"}
obv_argument = {"ObvSg", "ObvPl"}
prox_subj_marker = {"3SgProxSubj", "3PlProxSubj"}
prox_obj_marker = {"3SgProxObj", "3PlProxObj"}
obv_subj_marker = {"3SgObvSubj", "3PlObvSubj"}
obv_obj_marker = {"3SgObvObj", "3PlObvObj"}

# TODO: Check if inanimate obviatives are tagged with ProxSg/ObvSg etc.
#       I don't think the CGs fully handle these either...

prox_subj_count = prox_obj_count = obv_subj_count = obv_obj_count = 0
third_person_direct_verb_count = third_person_inverse_verb_count = 0

def is_third_person_transitive(tok):
    return is_third_person_direct(tok) or is_third_person_inverse(tok)

# e.g. 3SgProxSubj and 3PlObvObj
def is_third_person_direct(tok):
    xpos = tok.get("xpos")
    if not xpos: return False
    return any(m in xpos for m in prox_subj_marker) and any(m in xpos for m in obv_obj_marker)

# e.g. 3SgObvSubj and 3PlProxObj
def is_third_person_inverse(tok):
    xpos = tok.get("xpos")
    if not xpos: return False
    return any(m in xpos for m in obv_subj_marker) and any(m in xpos for m in prox_obj_marker)

with open(CORPUS_PATH, encoding="utf-8") as f:
    for sent in parse_incr(f):
        for tok in sent:
            xpos = tok.get("xpos")
            if not xpos:
                continue

            # argument counts
            if any(m in xpos for m in prox_argument) and tok["deprel"] == "nsubj" and is_third_person_transitive(sent[tok["head"]-1]):
                prox_subj_count += 1
            if any(m in xpos for m in prox_argument) and tok["deprel"] == "obj" and is_third_person_transitive(sent[tok["head"]-1]):
                prox_obj_count += 1
            if any(m in xpos for m in obv_argument) and tok["deprel"] == "nsubj":
                obv_subj_count += 1
            if any(m in xpos for m in obv_argument) and tok["deprel"] == "obj":
                obv_obj_count += 1

            # total (verb morphology) counts
            if is_third_person_direct(tok):
                third_person_direct_verb_count += 1
            if is_third_person_inverse(tok):
                third_person_inverse_verb_count += 1

print(
    f"Proximate subjects: {prox_subj_count}",
    f"Proximate objects: {prox_obj_count}",
    f"Obviative subjects: {obv_subj_count}",
    f"Obviative objects: {obv_obj_count}",
    sep="\n",
)


# TODO: I get 76 total arguments here, but 74 in the word_order.ipynb... 
#       need to figure out why the discrepancy. probably the logic doesn't match up 100% somewhere

TokenList<odaanaang, bimibatoowan, odayan, gaa-bimaagonebizod, ., metadata={sent_id: "1", text: "odaanaang bimibatoowan odayan gaa-bimaagonebizod .", eng: "No English translation."}>
TokenList<naawaakigan, ogii-sagamigoon, ezigaan, ., metadata={sent_id: "306", text: "naawaakigan ogii-sagamigoon ezigaan .", eng: "No English translation."}>
TokenList<gii-kwiinawi-inendam, gii-pi-wiindamawind, gichi-aakozinid, ookomisan, ., metadata={sent_id: "328", text: "gii-kwiinawi-inendam gii-pi-wiindamawind gichi-aakozinid ookomisan .", eng: "No English translation."}>
TokenList<babiiwizhenyiwa', oniijaanisa', ., metadata={sent_id: "376", text: "babiiwizhenyiwa' oniijaanisa' .", eng: "No English translation."}>
TokenList<ogii-wiikomaan, ji-bi-gaagiigidonid, iniwen, akiwenziiyan, ., metadata={sent_id: "919", text: "ogii-wiikomaan ji-bi-gaagiigidonid iniwen akiwenziiyan .", eng: "No English translation."}>
TokenList<biinjitawag, gii-piindoodewan, iniwen, zagimen, ., metadata={sent_id: "969", text: "bi

In [17]:
# create stats tables
import pandas as pd

stats = [
    {"argument type": "proximate subject", "total": third_person_direct_verb_count, "overt": prox_subj_count,},
    {"argument type": "obviative object", "total": third_person_direct_verb_count,  "overt": obv_obj_count,},
    {"argument type": "proximate object", "total": third_person_inverse_verb_count,  "overt": prox_obj_count,},
    {"argument type": "obviative subject", "total": third_person_inverse_verb_count, "overt": obv_subj_count,},
]
df = pd.DataFrame(stats)
df["covert"] = df["total"] - df["overt"]
df["overt_ratio"] = df["overt"] / df["total"]

print(df.to_string(index=False))

    argument type  total  overt  covert  overt_ratio
proximate subject    436     41     395     0.094037
 obviative object    436    312     124     0.715596
 proximate object     15      3      12     0.200000
obviative subject     15     47     -32     3.133333
